In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import numpy as np

# Load datasets
Data1 = pd.read_excel('original augumented dataset - Copy (2).xlsx', sheet_name='Dataset')
Data2 = pd.read_excel('original augumented dataset - Copy (2).xlsx', sheet_name='Sheet1')

# Display all columns for both DataFrames to check for consistency
pd.set_option('display.max_columns', None)
print("First 5 rows of Data1:")
print(Data1.head())
print("First 5 rows of Data2:")
print(Data2.head())

# Check column consistency before merging
expected_columns = ["address", "bank", "atm_id", "coord_x", "coord_y", "freguesia", "parish", "mway_dist", "police_dis", 
                    "income", "density", "age", "unp_rate", "com_dens", "event_date", "freq_attack", "success_attack"]
missing_cols_data1 = set(expected_columns) - set(Data1.columns)
missing_cols_data2 = set(expected_columns) - set(Data2.columns)

if missing_cols_data1:
    print(f"Warning: Data1 is missing the following columns: {missing_cols_data1}")
if missing_cols_data2:
    print(f"Warning: Data2 is missing the following columns: {missing_cols_data2}")

# Ensure the date column is in datetime format
Data1['event_date'] = pd.to_datetime(Data1['event_date'], errors='coerce')
Data2['event_date'] = pd.to_datetime(Data2['event_date'], errors='coerce')

# Function to forecast using linear regression based on historical time series data
def linear_regression_forecast(data, target_variable, features):
    # Prepare the data for training
    data = data.dropna(subset=[target_variable] + features)
    
    X = data[features]
    y = data[target_variable]
    
    # Check if enough data points exist
    if len(X) < 2:
        print(f"Not enough data to forecast {target_variable}. Using the last known value.")
        return y.iloc[-1] if not y.empty else np.nan

    # Linear regression model
    model = LinearRegression()
    model.fit(X, y)
    
    return model

# List of variables to forecast and the features used to predict them
variables_to_forecast = {
    'income': ['mway_dist', 'police_dis', 'density', 'age', 'unp_rate', 'com_dens'],
    'density': ['mway_dist', 'police_dis', 'income', 'age', 'unp_rate', 'com_dens'],
    'age': ['mway_dist', 'police_dis', 'income', 'density', 'unp_rate', 'com_dens'],
    'unp_rate': ['mway_dist', 'police_dis', 'income', 'density', 'age', 'com_dens'],
    'com_dens': ['mway_dist', 'police_dis', 'income', 'density', 'age', 'unp_rate']
}

# Train models for each variable
models = {var: linear_regression_forecast(Data1, var, features) for var, features in variables_to_forecast.items()}

# Apply the models to predict values for each row in Data2
for var, model in models.items():
    if model is not None:
        Data2[var] = model.predict(Data2[variables_to_forecast[var]].fillna(0))

# Cast specific columns to integer
columns_to_cast = ['income', 'age', 'unp_rate', 'com_dens']
Data2[columns_to_cast] = Data2[columns_to_cast].round().astype(int)

# Save the updated dataset
Data2.to_csv('adjusted_new_data.csv', index=False)

# Ensure both datasets have the correct columns before merging
Data1 = Data1[expected_columns]
Data2 = Data2[expected_columns]

# Append Data2 to Data1 and combine
combined_data = pd.concat([Data1, Data2], ignore_index=True)

# Sort by frequency of attacks for better analysis
combined_data = combined_data.sort_values(by='freq_attack', ascending=False)

# Reset index to maintain proper sequence
combined_data.reset_index(drop=True, inplace=True)

# Display the first 500 rows to validate the result
print("First 500 rows of the combined dataset:")
print(combined_data.head(500))

# Save the final combined data
combined_data.to_csv('CompleteData2024.csv', index=False)

print("Data processing completed and saved to CompleteData2024_v2.csv")


First 5 rows of Data1:
                           address                      bank atm_id  coord_x  \
0          Rua Junqueira 94 Lisbon       BANCO BIC PORTUGUES   BP12 -9.18528   
1  Rua Luis De Camoes 106-C Lisbon            BANCO BPI S.A.  BPI12 -9.18224   
2  Rua Luis De Camoes 106-C Lisbon            BANCO BPI S.A.  BPI82 -9.18224   
3   Rua Prior Do Crato 70 a Lisbon  CAIXA GERAL DE DEPOSITOS  CGD13 -9.17216   
4       Rua Da Junqueira 86 Lisbon  CAIXA GERAL DE DEPOSITOS  CGD61 -9.18457   

    coord_y  freguesia  parish    mway_dist  police_dis  income  density  age  \
0  38.69956  Alcântara       1  1233.199547  839.453359      48   0.2558   32   
1  38.70349  Alcântara       1   723.041603  407.387599      44  24.1571   30   
2  38.70349  Alcântara       1   723.041603  407.387599      44  24.1571   30   
3  38.70589  Alcântara       1   599.386780  501.757963      30  13.7788   50   
4  38.69984  Alcântara       1  1174.746261  771.670349      44   0.2558   27   

   unp_ra